In [1]:
pip install gurobipy

Note: you may need to restart the kernel to use updated packages.


In [4]:
from itertools import combinations
import math

def ler_instancia(arquivo_txt, g):
    with open(arquivo_txt, 'r') as f:
        linhas = [linha.strip() for linha in f if linha.strip()]

    # Primeira linha: número de operações, relações de precedência e máquinas
    o, p, n = map(int, linhas[0].split())

#pri, fri
    pri = [[] for _ in range(o)]  # predecessoras de i
    fri = [[] for _ in range(o)]  # sucessoras de i
    for linha in linhas[1:1+p]:
        i1, i2 = map(int, linha.split())
        pri[i2].append(i1)
        fri[i1].append(i2)

#Ri, pik
    Ri = [[] for _ in range(o)]
    pik = [[0 for _ in range(n)] for _ in range(o)]    
    linha_atual = 1 + p
    for i in range(o):
        dados = list(map(int, linhas[linha_atual + i].split()))
        m = dados[0]
        maquinas_tempos = dados[1:]
        for j in range(m):
            k = maquinas_tempos[2*j]
            pik[i][k] = maquinas_tempos[2*j + 1]
            Ri[i].append(k)
    
    linha_atual += o

#Of
    Of = [i for i in range(o) if not fri[i]]

#ddi    
    #dd: data de entrega para cada job    
    FFp = linhas[1 + p + o]
    #print(f'FFp {FFp}')
    valores = FFp.strip().split()
    valores_numericos = [v for v in valores if v != '.' and v != '']
    datas = list(map(float, valores_numericos))
    #print(f'datas {datas}')

    #g=0.25
    datas = [math.floor(x * g) for x in datas]
    #print(f'datas*g {datas}')
    
    ddi=[0]* len(range(o))
    for v, t in zip(Of, datas):
        ddi[v] = t
    #print(f'due date {ddi}')

#H    
    H = sum(pik[i - 1][k - 1] for i in range(o) for k in range(m))

#Pi e Fi
    Pij=[]
    for predecessores in pri:
        if len(predecessores) >= 2:
            pares = combinations(predecessores, 2)
            Pij.extend([list(p) for p in pares])
    Fij=[]
    for sucessores in fri:
        if len(sucessores) >= 2:
            pares = combinations(sucessores, 2)
            Fij.extend([list(p) for p in pares])

    return n, o, Ri, pri, fri, Of, pik,ddi,H,Pij, Fij

#caminho_arquivo = "C:\\Users\\usuario\\Doutorado\\Instâncias\\non-linear routes\\Y-type\\miniYFJS04.txt"
#n, o, Ri, pri, fri, Of, pik,ddi,H, Pij, Fij = ler_instancia(caminho_arquivo)


In [5]:
def resolver_modelo(n, o, Ri, pri, fri, Of, pik, ddi, H, Pij, Fij, nome_saida, DIR_SAIDA):
    import gurobipy as gp
    import os
    
  
    R=[k for k in range(n)] #conjunto de máquinas (k e M)
    O=[j for j in range(o)] #conjunto de operações (i,j e O)
    
    model=gp.Model()
    
    #VARIÁVEIS
    ti=model.addVars(O, name="t") #instante de início da operação i
    Ci=model.addVars(O, name="C") #completion time - instante de término da operação i - NOVO!
    aik=model.addVars(O,R,vtype=gp.GRB.BINARY, name="a") #1, se a operação i é processada pela máquina k; 0, caso contrário
    bij=model.addVars(O,O,vtype=gp.GRB.BINARY, name="b") #1 se i vem antes da operação j; 0, caso contrário
    Ti=model.addVars(Of, name="T") #tardiness da operação i - NOVO
    
    #FUNÇÃO OBJETIVO
    model.setObjective(gp.quicksum(Ti[i] for i in Of), sense=gp.GRB.MINIMIZE) 
            
    #RESTRIÇÕES
    r2=model.addConstrs(gp.quicksum(aik[i,k] for k in Ri[i])==1 for i in O) #garante que cada operação i deve ser processada por uma única máquina do conjunto  ok
    r3=model.addConstrs(ti[i]>=ti[j]+gp.quicksum(pik[j][k]*aik[j,k] for k in Ri[j]) for i in O for j in pri[i]) # define a precedência entre operações de um mesma tarefa
    r15=model.addConstrs(ti[i]>=ti[j]+pik[j][l]-(2-aik[i,k]-aik[j,l]+bij[i,j])*H 
                                 for i in O for j in O for k in Ri[i] for l in Ri[j] 
                                 if i!=j if k==l if j in O)
    r17=model.addConstrs(ti[j]>=ti[i]+pik[i][k]-(3-aik[i,k]-aik[j,l]-bij[i,j])*H 
                                 for i in O for j in O for k in Ri[i] for l in Ri[j] 
                                 if i!=j if k==l if i in O)      
    rCi=model.addConstrs(Ci[i]>=ti[i]+gp.quicksum(pik[i][k]*aik[i,k] for k in Ri[i]) for i in O) #calculao instante de término das últimas operações
    rTi=model.addConstrs(Ti[i]>=Ci[i]-ddi[i] for i in Of) #define o atraso das últimas operações ok
    rP=model.addConstrs(Ci[i]==Ci[j] for (i,j) in Pij)
    rF=model.addConstrs(ti[i]==ti[j] for (i,j) in Fij)

    model.update()
    #model.display()
   
    print(f"Número de variáveis: {model.NumVars}")
    print(f"Número de restrições: {model.NumConstrs}")

    # Criar diretório de saída se não existir
    os.makedirs(DIR_SAIDA, exist_ok=True)
    
    caminho_lp = os.path.join(DIR_SAIDA, f"{nome_saida}.lp")
    
    try:
        with open(caminho_lp, 'w') as f:
            f.write("teste")
        os.remove(caminho_lp)
        print("Escrita no diretório OK")
    except Exception as e:
        print(f"Erro de permissão: {e}")


        
    model.setParam('TimeLimit', 3600)
    #model.optimize()
    model.write(caminho_lp)    

    return {
        'modelo': model,
        'ti': ti,
        'Ci': Ci,
        'aik': aik,
        'bij': bij,
        'Ti': Ti
        }
    
    #print("Função objetivo:", model.ObjVal)
    #print("Número de variáveis:", model.NumVars)
    #print("Número de variáveis inteiras ou binárias:", sum(1 for v in model.getVars() if v.VType in ['I', 'B']))
    #print("Número de restrições:", model.NumConstrs)
    #print("Tempo de processamento (segundos):", model.Runtime)
    



In [7]:
import os
import glob
from datetime import datetime



def salvar_resultados_txt(m, caminho, O, Of, Ri):
    with open(caminho, 'w') as f:
        
        f.write(f"Status: {m.status}\n")
        f.write(f"Objetivo: {m.ObjVal}\n")
        f.write(f"Bound: {m.ObjBound}\n")
        f.write(f"Tempo: {m.Runtime:.2f}s\n")
        f.write(f"GAP: {m.MIPGap*100:.2f}%\n")
        f.write(f"# Variáveis: {m.NumVars}\n")
        f.write(f"# Variáveis inteiras: {sum(1 for v in model.getVars() if v.VType in ['I', 'B'])}\n")
        f.write(f"# Restrições: {model.NumConstrs}\n")

        
        f.write("\nInstante de inicio\n")
        for i in O:
            f.write(f"t[{i}] = {m.getVarByName(f't[{i}]').x}\n")
        
        f.write("\nCompletion time e atraso\n")
        for i in Of:
            f.write(f"T[{i}] = {m.getVarByName(f'T[{i}]').x}\n")
            f.write(f"C[{i}] = {m.getVarByName(f'C[{i}]').x}\n")

        f.write("\nAlocações:\n")
        for i in O:
            for k in Ri[i]:
                if m.getVarByName(f'a[{i},{k}]').x > 0.5:
                    f.write(f"Op{i} - M{k}\n")


DIR_INSTANCIAS = "C:\\Users\\Hanna Pamplona\\Doutorado\\Instâncias\\non-linear routes\\2. com FIFO\\DA_FFp"
DIR_SAIDA = "C:\\temp"

valores_g = [0.25, 0.5, 0.75]
tipos = ["tight", "moderate", "loose"]

for g, tipo in zip(valores_g, tipos):
    print(f"\n=== Processando g={g}, tipo={tipo} ===")

    for arquivo in glob.glob(os.path.join(DIR_INSTANCIAS, "*.txt")):
        nome_base = os.path.basename(arquivo).replace("_FFp.txt", "")
        nome_saida = f"MILP_Peres_{nome_base}_{tipo}_sem_block"
        print(f"\nProcessando: {nome_saida}")
        
        try:
            # 1. Ler instância
            n, o, Ri, pri, fri, Of, pik, ddi, H, Pij, Fij = ler_instancia(arquivo, g)
            O = range(o)
            R = range(n)
            
            # 2. Resolver modelo 
            resultados = resolver_modelo(n, o, Ri, pri, fri, Of, pik, ddi, H, Pij, Fij, nome_saida, DIR_SAIDA)
            
            print(f"Finalizado: {nome_saida}")
            
        except Exception as e:
            print(f"Erro em {nome_saida}: {str(e)}")


=== Processando g=0.25, tipo=tight ===

Processando: MILP_Peres_DAFJS01_tight_sem_block
Número de variáveis: 862
Número de restrições: 2648
Escrita no diretório OK
Set parameter TimeLimit to value 3600
Finalizado: MILP_Peres_DAFJS01_tight_sem_block

Processando: MILP_Peres_DAFJS02_tight_sem_block
Número de variáveis: 804
Número de restrições: 2480
Escrita no diretório OK
Set parameter TimeLimit to value 3600
Finalizado: MILP_Peres_DAFJS02_tight_sem_block

Processando: MILP_Peres_DAFJS03_tight_sem_block
Número de variáveis: 3691
Número de restrições: 15313
Escrita no diretório OK
Set parameter TimeLimit to value 3600
Finalizado: MILP_Peres_DAFJS03_tight_sem_block

Processando: MILP_Peres_DAFJS04_tight_sem_block
Número de variáveis: 2372
Número de restrições: 9620
Escrita no diretório OK
Set parameter TimeLimit to value 3600
Finalizado: MILP_Peres_DAFJS04_tight_sem_block

Processando: MILP_Peres_DAFJS05_tight_sem_block
Número de variáveis: 1803
Número de restrições: 4408
Escrita no dire

In [16]:
def salvar_gantt(resultados, caminho):
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    from collections import defaultdict, deque

    O = resultados['O']
    Of = resultados['Of']
    Ri = resultados['Ri']
    pri = resultados['pri']
    pik = resultados['pik']
    aik = resultados['aik']
    ti = resultados['ti']

    A = []
    for i, predecessores in enumerate(pri):
        for j in predecessores:
            A.append((j, i))
    # Liga operações sem predecessores ao dummy 0
    for i in range(len(pri)):
        if not pri[i]:
            A.append((0, i))
    
    def identificar_jobs_por_Of(Of):
        job_por_operacao = {}
        inicio = 0
        for job_id, fim in enumerate(Of, 1):  # jobs começam em 1
            for i in range(inicio, fim + 1):
                job_por_operacao[i] = job_id
            inicio = fim + 1
        return job_por_operacao
    
    job_por_operacao = identificar_jobs_por_Of(Of)
    
    cores_disponiveis = [
        'tab:blue', 'tab:orange', 'tab:green', 'tab:red',
        'tab:purple', 'tab:brown', 'tab:pink', 'tab:olive',
        'tab:cyan', 'tab:gray', 'gold', 'deeppink', 'limegreen'
    ]
    
    cores_job = {}
    for job in sorted(set(job_por_operacao.values())):
        cores_job[job] = cores_disponiveis[(job - 1) % len(cores_disponiveis)]
    

    escala = 1000  # ajustável
    dados_gantt = []
    dados_blocking = []
    
    for i in O:
        for k in Ri[i]:
            if aik[i, k].x == 1:
                inicio = ti[i].x
                fim = inicio + pik[i][k]
                dados_gantt.append((i, k, inicio, fim))
    
    # Blocking (com base em A, independente da máquina de j)
    for i, j in A:
        if i == 0:
            continue  # ignora dummy
    
        maq_i = next((k for k in Ri[i] if aik[i, k].x == 1), None)
        if maq_i is None:
            continue
    
        fim_i = ti[i].x + pik[i][maq_i]
        ini_j = ti[j].x
    
        if ini_j > fim_i:
            dados_blocking.append((maq_i, fim_i, ini_j))
    
    # =====================
    # === Plotagem ===
    # =====================
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Operações principais
    for op, maq, ini, fim in dados_gantt:
        ini /= escala
        fim /= escala
        job = job_por_operacao.get(op, 0)
        cor = cores_job.get(job, 'tab:gray')  
        ax.add_patch(
            patches.Rectangle((ini, maq - 0.4), fim - ini, 0.8,
                              edgecolor='black',
                              facecolor=cor)
        )
        ax.text((ini + fim) / 2, maq, f"{op}", ha='center', va='center', color='white', fontsize=9)
    
    # Retângulos de blocking
    for maq, ini, fim in dados_blocking:
        ini /= escala
        fim /= escala
        ax.add_patch(
            patches.Rectangle((ini, maq - 0.4), fim - ini, 0.8,
                              edgecolor='gray',
                              facecolor='lightgray',
                              linestyle='--',
                              linewidth=1.5)
        )
    
    # Eixos
    maquinas_usadas = sorted(set(maq for _, maq, _, _ in dados_gantt))
    ax.set_yticks(maquinas_usadas)
    ax.set_yticklabels([f"M{k}" for k in maquinas_usadas])
    ax.set_ylim(min(maquinas_usadas) - 0.5, max(maquinas_usadas) + 0.5)
    ax.set_xlabel("Tempo")
    ax.set_ylabel("Máquinas")
    ax.set_title("Gráfico de Gantt - DPeres^TT")
    ax.grid(True)
    plt.tight_layout()
    plt.savefig(caminho, dpi=300, bbox_inches='tight')
    plt.close()


###### 